# Rubix Step-by-Step Science Verification Notebook

This notebook verifies the main pieces of the science workflow in a controlled synthetic setup:

1. Gradient correctness on **multi-spaxel, multi-particle** data.
2. Finite-difference comparison against autodiff gradients.
3. Optimization diagnostics (loss and gradient norms).
4. Variational inference diagnostics (objective, reconstruction, KL).
5. Posterior predictive + residual product sanity checks.

The goal is to provide a reproducible diagnostic workflow that can be adapted to real runs and converted into paper figures.


## 0. Imports and Plot Style

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
# optional:
os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from rubix.core.data import Galaxy, GasData, RubixData, StarsData
from rubix.inference import (
    compare_gradients,
    finite_difference_grad,
    optimize_ifu_cube,
    optimize_variational_ifu_cube,
    sample_posterior_predictive_cubes,
    summarize_predictive_cube_samples,
    compute_residual_products,
    summarize_masked_metrics,
)

plt.style.use('default')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

## 1. Build a Multi-Particle, Multi-Spaxel Synthetic Pipeline

We define a small synthetic forward model with:

- `n_particles > 1`
- `n_spaxel_x * n_spaxel_y > 1`
- `n_wave > 1`

Each particle contributes to every voxel through a smooth differentiable basis; this allows robust gradient testing.


In [ ]:
class MultiParticleIFUSyntheticPipeline:
    """Differentiable synthetic IFU forward model for diagnostics."""

    def __init__(self, n_particles=6, nx=8, ny=8, nw=32, seed=0):
        self.n_particles = n_particles
        self.nx = nx
        self.ny = ny
        self.nw = nw

        key = jax.random.PRNGKey(seed)
        k1, k2, k3 = jax.random.split(key, 3)

        # Basis cube per particle: shape (P, X, Y, W)
        # Each particle contributes a smooth, nonlinear function of its age and metallicity, multiplied by this basis cube.
        # This toy particle_basis is deliberately over-general to create a hard test.
        self.particle_basis = jax.random.normal(k1, (n_particles, nx, ny, nw)) * 0.2

        # Smooth positive weights used to combine age + metallicity effects
        self.age_weight = jnp.abs(jax.random.normal(k2, (n_particles, 1, 1, 1))) + 0.2
        self.met_weight = jnp.abs(jax.random.normal(k3, (n_particles, 1, 1, 1))) + 0.2

    def run_sharded(self, rubixdata: RubixData) -> jnp.ndarray:
        age = rubixdata.stars.age.reshape(self.n_particles, 1, 1, 1)
        metallicity = rubixdata.stars.metallicity.reshape(self.n_particles, 1, 1, 1)

        # Nonlinear, smooth contribution per particle
        # artificially combines age and metallicity effects in a non-additive way, but is still fully differentiable
        # mainly intended to test that gradients can flow through a non-trivial forward model, and that optimization can succeed
        contrib = (
            jnp.tanh(age * self.age_weight)
            + jnp.sqrt(jnp.clip(metallicity, 1e-8, None)) * self.met_weight
        ) * self.particle_basis

        cube = jnp.sum(contrib, axis=0)

        if rubixdata.noise_key is not None:
            noise = jax.random.normal(rubixdata.noise_key, shape=cube.shape, dtype=cube.dtype)
            cube = cube + 0.01 * noise

        return cube


def make_static_data(n_particles: int) -> RubixData:
    return RubixData(
        galaxy=Galaxy(),
        stars=StarsData(
            coords=jnp.zeros((n_particles, 3)),
            velocity=jnp.zeros((n_particles, 3)),
            mass=jnp.ones(n_particles),
            age=jnp.ones(n_particles),
            metallicity=jnp.ones(n_particles) * 0.02,
        ),
        gas=GasData(
            coords=jnp.zeros((1, 3)),
            velocity=jnp.zeros((1, 3)),
            mass=jnp.ones(1),
        ),
    )

## 2. Create Ground Truth and Initial Guess

In [ ]:
n_particles = 6
nx, ny, nw = 8, 8, 32

pipe = MultiParticleIFUSyntheticPipeline(
    n_particles=n_particles,
    nx=nx,
    ny=ny,
    nw=nw,
    seed=42,
)

static_data = make_static_data(n_particles)

true_params = {
    'stars': {
        'age': jnp.linspace(0.6, 2.0, n_particles),
        'metallicity': jnp.linspace(0.005, 0.03, n_particles),
    }
}

init_params = {
    'stars': {
        'age': jnp.ones(n_particles) * 1.2,
        'metallicity': jnp.ones(n_particles) * 0.015,
    }
}

target_cube = pipe.run_sharded(
    RubixData(
        galaxy=static_data.galaxy,
        stars=StarsData(
            coords=static_data.stars.coords,
            velocity=static_data.stars.velocity,
            mass=static_data.stars.mass,
            age=true_params['stars']['age'],
            metallicity=true_params['stars']['metallicity'],
        ),
        gas=static_data.gas,
    )
)

print('target shape:', target_cube.shape)
print('target min/max:', float(target_cube.min()), float(target_cube.max()))

## 3. Gradient Verification: Autodiff vs Finite Differences

We define a scalar objective on the full IFU cube and compare JAX gradients vs central finite-difference gradients.


In [ ]:
def objective(params):
    pred = pipe.run_sharded(
        RubixData(
            galaxy=static_data.galaxy,
            stars=StarsData(
                coords=static_data.stars.coords,
                velocity=static_data.stars.velocity,
                mass=static_data.stars.mass,
                age=params['stars']['age'],
                metallicity=params['stars']['metallicity'],
            ),
            gas=static_data.gas,
        )
    )
    return jnp.mean((pred - target_cube) ** 2)

val, grads_auto = jax.value_and_grad(objective)(init_params)
grads_fd = finite_difference_grad(objective, init_params, eps=1e-4)
summary = compare_gradients(grads_auto, grads_fd)

print('objective:', float(val))
print('max_abs_error:', float(summary.max_abs_error))
print('relative_l2_error:', float(summary.relative_l2_error))

In [ ]:
g_auto_age = np.asarray(grads_auto['stars']['age'])
g_fd_age = np.asarray(grads_fd['stars']['age'])

g_auto_met = np.asarray(grads_auto['stars']['metallicity'])
g_fd_met = np.asarray(grads_fd['stars']['metallicity'])

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(g_auto_age, g_fd_age)
lims = [min(g_auto_age.min(), g_fd_age.min()), max(g_auto_age.max(), g_fd_age.max())]
axs[0].plot(lims, lims, 'k--', alpha=0.7)
axs[0].set_title('Age gradient: autodiff vs finite diff')
axs[0].set_xlabel('autodiff')
axs[0].set_ylabel('finite diff')

axs[1].scatter(g_auto_met, g_fd_met)
lims = [min(g_auto_met.min(), g_fd_met.min()), max(g_auto_met.max(), g_fd_met.max())]
axs[1].plot(lims, lims, 'k--', alpha=0.7)
axs[1].set_title('Metallicity gradient: autodiff vs finite diff')
axs[1].set_xlabel('autodiff')
axs[1].set_ylabel('finite diff')

plt.tight_layout()
plt.show()

## 4. Deterministic Optimization Diagnostics

We run `optimize_ifu_cube` and inspect convergence behavior.


In [ ]:
mask = jnp.ones_like(target_cube)
weights = jnp.ones_like(target_cube)

opt_result = optimize_ifu_cube(
    pipeline=pipe,
    params_init=copy.deepcopy(init_params),
    static_data=static_data,
    target=target_cube,
    mask=mask,
    weights=weights,
    learning_rate=5e-2,
    max_steps=1000,
    tol=1e-8,
)

print('final_loss:', opt_result.final_loss)
print('best_loss:', opt_result.best_loss)
print('steps_run:', opt_result.steps_run)
print('converged:', opt_result.converged)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(opt_result.loss_history)
axs[0].set_yscale('log')
axs[0].set_title('Optimization loss history')
axs[0].set_xlabel('step')
axs[0].set_ylabel('loss')

axs[1].plot(opt_result.grad_norm_history)
axs[1].set_yscale('log')
axs[1].set_title('Optimization grad-norm history')
axs[1].set_xlabel('step')
axs[1].set_ylabel('||grad||')

plt.tight_layout()
plt.show()

## 5. Variational Inference Diagnostics

We run `optimize_variational_ifu_cube` and inspect objective decomposition.


In [ ]:
sigma = jnp.ones_like(target_cube)

vi_result = optimize_variational_ifu_cube(
    pipeline=pipe,
    params_init=copy.deepcopy(init_params),
    static_data=static_data,
    target=target_cube,
    sigma=sigma,
    learning_rate=2e-2,
    max_steps=500,
    tol=1e-8,
    num_samples=3,
    beta_kl=1e-3,
    seed=0,
)

print('final_objective:', vi_result.final_objective)
print('final_reconstruction:', vi_result.final_reconstruction)
print('final_kl:', vi_result.final_kl)
print('steps_run:', vi_result.steps_run)
print('converged:', vi_result.converged)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(vi_result.objective_history)
axs[0].set_title('VI objective history')
axs[0].set_xlabel('step')
axs[0].set_ylabel('objective')

axs[1].plot(vi_result.reconstruction_history)
axs[1].set_title('VI reconstruction history')
axs[1].set_xlabel('step')
axs[1].set_ylabel('reconstruction')

axs[2].plot(vi_result.kl_history)
axs[2].set_title('VI KL history')
axs[2].set_xlabel('step')
axs[2].set_ylabel('KL')

plt.tight_layout()
plt.show()

## 6. Posterior Predictive and Residual Diagnostics

In [ ]:
samples = sample_posterior_predictive_cubes(
    pipeline=pipe,
    posterior_mean_params=vi_result.posterior_mean_params,
    posterior_log_std_params=vi_result.posterior_log_std_params,
    static_data=static_data,
    num_samples=12,
    seed=1,
)
summary_pred = summarize_predictive_cube_samples(samples)
residuals = compute_residual_products(
    prediction=summary_pred['mean'],
    target=target_cube,
    sigma=sigma,
)
metrics = summarize_masked_metrics(
    prediction=summary_pred['mean'],
    target=target_cube,
    mask=mask,
)

print(metrics)

In [ ]:
wave_idx = nw // 2

fig, axs = plt.subplots(1, 4, figsize=(18, 4))

im_t = axs[0].imshow(np.asarray(target_cube[:, :, wave_idx]))
axs[0].set_title(f'Ground truth slice (wave={wave_idx})')
plt.colorbar(im_t, ax=axs[0], fraction=0.046)

im0 = axs[1].imshow(np.asarray(summary_pred['mean'][:, :, wave_idx]))
axs[1].set_title(f'Posterior mean slice (wave={wave_idx})')
plt.colorbar(im0, ax=axs[1], fraction=0.046)

im1 = axs[2].imshow(np.asarray(residuals['residual'][:, :, wave_idx]),cmap='coolwarm')
axs[2].set_title('Residual slice')
plt.colorbar(im1, ax=axs[2], fraction=0.046)

im2 = axs[3].imshow(np.asarray(residuals['chi2'][:, :, wave_idx]))
axs[3].set_title('Chi2 slice')
plt.colorbar(im2, ax=axs[3], fraction=0.046)

plt.tight_layout()
plt.show()


## 7. What to Check Before Trusting Real-Data Results

- Gradient agreement: small `max_abs_error` and low `relative_l2_error`.
- Optimization: monotonic/mostly decreasing loss, no exploding grad norm.
- VI: objective decreases; reconstruction and KL are numerically stable.
- Residuals: no strong structured artifacts; chi2 map does not show systematic hotspots.

Use this notebook as a template for real IFU runs by replacing the synthetic pipeline/data with your real run outputs and config.
